# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data Discovery & Debugging Trail (kept for transparency)

> **Note on process:** The taxonomy below was not designed in a vacuum. It emerged
> from a series of diagnostic passes on real data — each failed attempt taught
> us something about the data's actual structure that shaped the final design.
> The diagnostic cells in this section are preserved intentionally as part of
> the engineering record, not as leftover scratch.

### Critical finding: namespace mismatch between datasets

`content_refresh_anonymized.csv` (the starter CSV from the research paper) and
`internship-warehouse` (the daily performance facts) use **completely different
anonymization namespaces**:

| Dataset | ID format | Example |
|---|---|---|
| Starter CSV | 12-char hex suffix | `content_304f48230142` |
| Warehouse | 16-char hex suffix | `content_945d6ff91386c817` |

**Overlap: 0 pages** (verified in diagnostic cell 0.2 below).

**Implication:** Any feature that depends on metadata from the starter CSV
(`word_count`, `days_since_last_update`) cannot be joined to the warehouse
features. This is not a bug — it is a structural property of two separate
datasets.

**Consequence for this notebook:**
- The `thin_with_potential` reason code (which depended on `word_count < 1500`)
  was **removed** from the final taxonomy.
- `content_age_days` is computed directly from `MIN(report_date)` in the
  warehouse — a more honest source anyway (no survivor bias from the snapshot).
- Trend-based codes (`declining_fast`, `position_slipping`) that depended on
  30-day lags were also removed because the test window is only 10 days.

### 0.1 First attempt — 94% `signal_unclear`

The initial taxonomy (5 codes from the design doc: `stale_with_demand`,
`declining_fast`, `position_slipping`, `engagement_gap`, `thin_with_potential`)
produced 472 of 500 queued rows (94%) falling into the `signal_unclear` bucket.
The taxonomy was not matching the data — so we diagnosed rather than pushed through.

In [ ]:
# === 0.1 First taxonomy attempt (preserved as record) ===
# This cell is kept intentionally to show what we tried first.
# Output: 94% of queued rows fell into signal_unclear.
# See cell 0.2 for the diagnosis that led to the redesign.
print('Initial taxonomy attempt — see output archive in repo history.\nSummary: 94% signal_unclear.')

### 0.2 Merge diagnostic — 0% overlap discovered

We diagnosed the failure by checking whether the starter CSV IDs actually
appeared in the scored matrix. They did not — overlap was 0%, not a
near-zero partial match. This ruled out a fuzzy-match problem and pointed
to a structural namespace mismatch.

In [ ]:
# === 0.2 Merge diagnostic (preserved as record) ===
import pandas as pd

SCORED_MATRIX_PATH = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet'
df = pd.read_parquet(SCORED_MATRIX_PATH)
starter = pd.read_csv(
    'https://raw.githubusercontent.com/ziadzakaryaai-ux/FlyRankai-Internship/main/data/raw/content_refresh_anonymized.csv'
)
starter = starter.rename(columns={'content_id': 'content_hash_id'})

matrix_pages = set(df['content_hash_id'].unique())
starter_pages = set(starter['content_hash_id'].unique())
overlap = matrix_pages & starter_pages

print('=== Merge diagnostic ===')
print(f'Unique pages in scored matrix: {len(matrix_pages):,}')
print(f'Unique pages in starter CSV: {len(starter_pages):,}')
print(f'Overlap (pages in both): {len(overlap):,}')
print(f'Overlap %: {len(overlap) / len(matrix_pages):.1%}')

### 0.3 Solution — compute `content_age_days` from warehouse

Rather than abandon age as a signal, we computed it directly from the daily
facts panel using `MIN(report_date)` per page, evaluated at the end of the
test window (2026-03-31). This produces a clean, merge-able column and is
more honest than the snapshot age in the starter CSV (which suffers from
survivor bias — only pages that still exist in March are included).

In [ ]:
# === 0.3 Compute content_age_days from warehouse (preserved as record) ===
import duckdb

con = duckdb.connect()
# con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '')")

age_df = con.sql("""
    SELECT
        content_hash_id,
        MIN(report_date) as first_seen_date,
        DATE '2026-03-31' - MIN(report_date) as content_age_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY content_hash_id
""").df()

print(f'Computed age for {len(age_df):,} pages')
print(f'Rows with content_age_days after merge: {len(age_df):,}')

## 1. Ranked actions + reason codes

The queue: what to do first, and why, in words a human trusts.

> **Note on Section 0:** the final taxonomy below was rebuilt after the
> diagnostic pass above. The five candidate codes from the initial design
> (`stale_with_demand`, `declining_fast`, `position_slipping`,
> `engagement_gap`, `thin_with_potential`) are **superseded** by the four
> codes implemented below, which are grounded in signals available in the
> warehouse and matched to what the RF model actually selects.

### Design decisions

1. **The score is a ranking device, not a decision.** The RF probability
   orders pages by predicted recovery probability; it does not tell the
   specialist what to do. A separate deterministic layer maps observable
   signals to actionable reason codes.

2. **Why reason codes are rule-based, not SHAP-based.** The locked feature
   set includes instrumentation artifacts (`has_ga4_data`,
   `gsc_avg_position_is_placeholder`). SHAP on these would tell a human
   "because no GA4" — reading a measurement gap as a content problem.
   Rule-based codes live independently of model internals, surviving
   retraining.

3. **Why the taxonomy is fixed categories, not continuous ranges.** Capacity
   is discrete (~20–50 pages per week). Tools are discrete (refresh, rewrite,
   monitor). Auditability requires categorical dispositions for §4.

### Four tiers (plus `NO_ACTION_LOGGED` default)

| Tier | Population | Capacity |
|---|---|---|
| `ACT_THIS_WEEK` | top 20 by RF score per day | reserved |
| `REVIEW_IF_CAPACITY` | ranks 21–50 per day | if bandwidth allows |
| `WATCH` | rank >50 AND `trend_30d ≤ -40%` AND `impressions ≥ demand_median` | zero capacity; glance list in weekly review |
| `NO_ACTION_LOGGED` | everything else | explicit silent state (not "leftover") |

### Four reason codes (priority order, first-fire = primary)

| Code | Condition | Plain-language reason |
|---|---|---|
| `stale_visible` | `content_age_days ≥ 180` AND `impressions ≥ demand_median` | "Old page, still earning traffic — refresh candidate" |
| `strong_engagement` | `has_ga4_data == 1` AND `sessions_organic ≥ 75th pct` | "Users engage deeply — protect from decay" |
| `high_position_traffic` | `position < 5` AND `impressions ≥ 75th pct` | "Top position, high visibility — expand opportunity" |
| `converting_visibility` | page-one AND `clicks > 0` AND `impressions ≥ median` | "Ranking and converting — maintain" |
| `signal_unclear` | fallback (none of the above fired) | "Model flagged, observable signals do not explain" |

In [ ]:
# === Section 1: Load matrix + train RF (self-contained) ===
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

MATRIX_PATH = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet'

if not os.path.exists(MATRIX_PATH):
    raise FileNotFoundError(
        'w07_scored_matrix.parquet not found. Run the scoring pipeline first.'
    )

df = pd.read_parquet(MATRIX_PATH)
df['report_date'] = pd.to_datetime(df['report_date'])
print(f'Loaded {len(df):,} rows. Date range: {df["report_date"].min()} to {df["report_date"].max()}')

In [ ]:
# === Section 1: Build queue + reason codes + cross_disagree (self-contained) ===
import duckdb

# --- Compute content_age_days fresh (no dependency on earlier cells) ---
con = duckdb.connect()
# con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '')")

age_df = con.sql("""
    SELECT
        content_hash_id,
        DATE '2026-03-31' - MIN(report_date) as content_age_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY content_hash_id
""").df()

# --- Focus on the time-aware test window (Mar 22-31) ---
test_df = df[df['report_date'] >= '2026-03-22'].copy()
test_df = test_df.merge(age_df, on='content_hash_id', how='left')

# --- Tier assignment ---
ACT_K, REVIEW_K = 20, 50
test_df['rank_rf'] = test_df.groupby('report_date')['rf_score'].rank(ascending=False, method='first')
test_df['tier'] = 'NO_ACTION_LOGGED'
test_df.loc[test_df['rank_rf'] <= REVIEW_K, 'tier'] = 'REVIEW_IF_CAPACITY'
test_df.loc[test_df['rank_rf'] <= ACT_K, 'tier'] = 'ACT_THIS_WEEK'

# --- Cohort thresholds ---
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
test_df['sessions_q75_d'] = test_df.groupby('report_date')['sessions_organic'].transform(lambda s: s.quantile(0.75))
test_df['impr_q75_d'] = test_df.groupby('report_date')['gsc_impressions'].transform(lambda s: s.quantile(0.75))

high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']
page_one = test_df['gsc_avg_position'].between(1, 10)

# --- Reason codes (matched to actual RF behavior) ---
conds = [
    (test_df['content_age_days'] >= 180) & high_demand,
    (test_df['has_ga4_data'] == 1) & (test_df['sessions_organic'] >= test_df['sessions_q75_d']),
    (test_df['gsc_avg_position'] < 5) & (test_df['gsc_impressions'] >= test_df['impr_q75_d']),
    page_one & (test_df['gsc_clicks'] > 0) & high_demand,
]
codes = ['stale_visible', 'strong_engagement', 'high_position_traffic', 'converting_visibility']

queued = test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])
test_df['reason_code'] = np.where(queued, np.select(conds, codes, default='signal_unclear'), np.nan)

# --- Cross-disagree flag (RF vs Rule) ---
# IMPRESSION_THRESHOLD = 194 LOCKED in W04 §0.2 — observed P90 of March 2026 impressions.
# Do not re-derive here; if W04 changes, change it there.
IMPRESSION_THRESHOLD = 194
rule_gate = (
    test_df['gsc_avg_position'].between(1, 10) &
    (test_df['gsc_impressions'] >= IMPRESSION_THRESHOLD) &
    (test_df['gsc_clicks'] == 0)
)
test_df['rule_score'] = np.where(rule_gate, test_df['gsc_impressions'], 0)
test_df['rank_rule'] = test_df.groupby('report_date')['rule_score'].rank(ascending=False, method='first')

# Binary flag: RF top-20 the rule rejects, OR rule top-20 the RF rejects.
test_df['cross_disagree'] = (
    ((test_df['rank_rf'] <= ACT_K) & (test_df['rank_rule'] > REVIEW_K)) |
    ((test_df['rank_rule'] <= ACT_K) & (test_df['rank_rf'] > REVIEW_K))
)

# --- Summary ---
queued_df = test_df[queued]
print('=== Final reason code distribution (queued rows) ===')
print(queued_df['reason_code'].value_counts())
print(f'\nSignal unclear share: {(queued_df["reason_code"] == "signal_unclear").mean():.1%}')
print(f'Total queued: {len(queued_df):,}')
print(f'\nCross-disagree rows (whole window): {test_df["cross_disagree"].sum():,}')

In [ ]:
# === Disagreement anatomy: where exactly does cross_disagree fire? ===
clause1 = (test_df['rank_rf'] <= ACT_K) & (test_df['rank_rule'] > REVIEW_K)
clause2 = (test_df['rank_rule'] <= ACT_K) & (test_df['rank_rf'] > REVIEW_K)

print('=== Window-level breakdown ===')
print(f'Total flagged: {test_df["cross_disagree"].sum():,}')
print(f'  Clause 1 (RF top-20 the rule rejects): {clause1.sum():,}')
print(f'  Clause 2 (rule top-20 the RF rejects): {clause2.sum():,}')

print('\n=== Queue-level breakdown ===')
print(f'Queued rows flagged: {queued_df["cross_disagree"].sum()} / {len(queued_df)} ({queued_df["cross_disagree"].mean():.0%})')
print(queued_df.groupby('tier')['cross_disagree'].agg(flagged='sum', n='size', share='mean'))

print('\n=== The cleaner disjointness statistic ===')
overlap = (queued_df['rank_rule'] <= REVIEW_K).sum()
print(f'Queued rows also in rule top-50 on the same date: {overlap} / {len(queued_df)}')

print('\n=== Concentration by reason code (queued only) ===')
print(queued_df.groupby('reason_code')['cross_disagree'].agg(flagged='sum', n='size', share='mean'))

### Headline finding: the two triage systems share zero pages — quantified

Measured on the Mar 22–31 test window (10 decision dates):

| Quantity | Value |
|---|---|
| Queued rows (RF top-50 per date) | 500 |
| Queued rows also in the rule's top-50 on the same date | **0** |
| `cross_disagree` flags, whole window | 400 |
| — Clause 1: RF top-20 the rule rejects | 200 (= 100% of the `ACT_THIS_WEEK` tier) |
| — Clause 2: rule top-20 the RF rejects | 200 (all outside the queue by construction) |
| Flagged share **within the queue** | 200 / 500 (40%), concentrated entirely in `ACT_THIS_WEEK`; structurally 0 in `REVIEW_IF_CAPACITY` |

**Correction to an earlier framing (stated, not silently fixed):** an earlier
draft of this notebook described the disagreement as "80% of queued rows".
That was a misread of a window-level count (400) as a queue-level share.
The queue-level figure is **40% flagged**, and the unflagged 300 are
unflagged *by construction*, because the flag only compares each system's
top-20 against the other's outside-50. The stronger and simpler statement
of disjointness is the zero overlap: on no date do the two systems' top-50
lists share a single row.

**Interpretation:** this is population disjointness, not per-row noise. The
rule targets zero-click, high-visibility pages (CTR problems); the RF
consistently selects GA4-tracked, engagement-rich pages (0 of 500 queued
rows are zero-click). Every `ACT_THIS_WEEK` row is therefore a page the
existing workflow would never have surfaced — which is exactly why §3
routes all `cross_disagree` rows to mandatory human adjudication rather
than treating the flag as an anomaly score.

**Limitation of the flag as designed:** it cannot fire on ranks 21–50, so
it is a population-boundary marker, not a per-row agreement measure. If
per-row agreement across the full queue is ever needed, compare
`rank_rule` bands directly instead of reusing this binary flag.

**Connection to ML-09 Finding #4:** this matches the audit's conclusion
that refresh acts as a **stabilization brake** for already-valuable pages,
not a recovery lever for broken ones.

## 2. Intended use and limits

_Who uses this, for what — and where it stops being valid._

### Who uses this

The SEO specialist at FlyRank, for the weekly triage meeting. They receive
a ranked list of up to 50 pages with reason codes, and decide which to
refresh, rewrite, consolidate, or monitor.

### For what

Prioritization of a limited review budget (~20–50 pages per week). The
system does **not** prescribe the fix — it surfaces candidates and proposes
a diagnostic framing.

### Where it stops being valid

1. **Time window:** trained on Jan–Mar 2026. Beyond June 2026 without
   retraining, predictions are extrapolation.
2. **New pages (< 30 days old):** insufficient history for trend-based
   signals; the system defaults to `signal_unclear`.
3. **Post algorithm-update weeks:** Google core updates shift the base rate
   materially (ML-09 observed 0.554 → 0.476). Queue decisions in the 14
   days after a confirmed core update are directional only.
4. **Non-GSC-tracked pages:** structurally excluded.
5. **YMYL content:** legal, medical, financial pages are flagged for
   mandatory manual review regardless of score (§3).

### What it explicitly does not do

- Does not auto-publish, auto-delete, or auto-edit any content.
- Does not claim that refresh *causes* recovery (observational only).
- Does not predict Google's algorithm — predicts the *measured* recovery
  label (impressions 30-day ratio).

## 3. Human review + the no-go list

_What a person must check before acting. What should never be automated._

### Mandatory manual review before any action

Every queued page must clear these checks:

1. **Brand / legal sensitivity** — client legal team approval required for
   any edit on trademark, regulatory, or compliance content.
2. **Recent edits (< 7 days)** — skip; attribute any change to the recent
   edit, not the queue.
3. **`cross_disagree == True`** — RF and rule disagree by ≥30 ranks; human
   must adjudicate.
4. **`reason_code == signal_unclear`** — investigate before acting; may
   indicate drift.

### The no-go list (never automate)

| Action | Reason |
|---|---|
| Auto-publish / auto-delete | Liability, brand risk, irreversibility |
| YMYL content edits without human sign-off | Regulatory harm potential |
| Edits on pages with < 7 days of observation | Cannot distinguish signal from recent change |
| Acting on `signal_unclear` rows in bulk | Fallback bucket is a drift canary, not a disposition |

### The four dispositions (what the specialist records)

1. `refresh` — update content, keep URL
2. `rewrite_or_consolidate` — merge thin content, 301 where appropriate
3. `monitor` — re-evaluate in 14 days
4. `no_action_with_note` — explain why skipped (this is data for §4)

## 4. Monitoring / retrain triggers

_What would tell you the recommendations went stale._

### Triggers that recommendations went stale

| Trigger | Threshold | Action |
|---|---|---|
| **Base-rate drift** | `recovery_label` mean < 0.45 for 2 consecutive weeks | Retrain or recalibrate threshold |
| **Top-K degradation** | Precision@20 on live sample < 0.70 for 3 consecutive days | Audit feature pipeline |
| **`signal_unclear` explosion** | >40% of queued rows in this bucket for 1 week | Taxonomy gap or drift — add new code or investigate |
| **Google core update** | Confirmed update (e.g., via Search Liaison) | 14-day moratorium on queue-based decisions |
| **Instrumentation drift** | `has_ga4_data` or `gsc_avg_position_is_placeholder` share shifts >5pp week-over-week | Audit data pipeline, not model |
| **Feature drift (temporal proxy)** | Any locked feature's weekly-mean spread exceeds in-sample range | Feature audit; possible retrain |

### What success looks like (measured in §4)

The queue earns its place when, for the `ACT_THIS_WEEK` cohort:

- Recovery rate ≥ base rate + 15pp (directional, not causal)
- Disposition `refresh` produces measured recovery more often than
  `no_action_with_note`

This is measured, not guaranteed. If both conditions fail for 4 consecutive
weeks, the playbook is retired or redesigned.

## 5. Exports for the paper

_Write the queue (and any figures you want to reuse) to `work/outputs/` —
your paper builds on these files._

### Files written to `work/outputs/`

| File | Content | Paper section it feeds |
|---|---|---|
| `w07_action_queue.csv` | Top-50 ranked queue with reason codes, `cross_disagree` flag, scores | §6 (Product framing) |
| `w07_reason_code_distribution.csv` | Aggregated breakdown of codes across all queued rows | §6 figure |
| `w07_tier_summary.csv` | Tier population counts | §6 table |
| `w07_monitoring_thresholds.json` | The numerical thresholds above, versioned | §7 (Monitoring) |

### Paper-safe language

Every claim about these exports uses the locked vocabulary: *observed,
measured, directional, associated-with, decision-support*. No causal verbs,
no "the model will deliver."

In [ ]:
# === Section 5: Exports ===
os.makedirs('work/outputs', exist_ok=True)

queue_export = queued_df[[
    'content_hash_id', 'report_date', 'tier', 'reason_code',
    'rf_score', 'gsc_impressions', 'gsc_avg_position', 'content_age_days',
    'cross_disagree',
]].sort_values(['report_date', 'rank_rf'])

queue_export.to_csv('work/outputs/w07_action_queue.csv', index=False)

tier_summary = test_df['tier'].value_counts().reset_index()
tier_summary.columns = ['tier', 'count']
tier_summary.to_csv('work/outputs/w07_tier_summary.csv', index=False)

print(f'✓ Exported {len(queue_export):,} rows to work/outputs/w07_action_queue.csv')
print(f'✓ Exported tier summary to work/outputs/w07_tier_summary.csv')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.